# 37장 · 청크·중첩·검색 개수를 한 변수씩 비교하기

검색 평가에서 기대한 원문을 찾았는데 답에 ‘14일’이 보이지 않습니다. 원문 한 쪽을 몇 글자씩 나누느냐에 따라 검색기에 보이는 조각이 달라집니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 원문의 어디서 잘렸나요?

여기서는 토큰 대신 파이썬 문자열의 글자 위치로 자릅니다. size는 한 조각의 최대 길이, overlap은 다음 조각 앞부분에 다시 들어갈 글자 수입니다. 먼저 크기만 바꾸어 첫 조각에 어떤 내용이 남았는지 읽습니다.



**볼 곳**　창이 오른쪽으로 이동한 뒤 온점이 어느 조각에 남는지 보세요.

**애니메이션의 의미**　16글자에서 자르면 온점이 다음 조각으로 넘어갑니다.


### 셀 01 · 길이 16으로 나눈 첫 조각

도서 대출 원문을 둘로 나눕니다. 첫 조각 안에 어떤 숫자가 있나요?

In [1]:
def split_text(text, size, overlap):
    if not 0 <= overlap < size:
        raise ValueError("중첩은 0 이상이고 크기보다 작아야 합니다.")
    parts = []
    for start in range(0, len(text), size - overlap):
        parts.append(text[start:start + size])
        if start + size >= len(text):
            break
    return parts

text = '도서 대출 기간은 14일입니다.'
print(split_text(text, 16, 0))

['도서 대출 기간은 14일입니다', '.']


첫 조각에는 14일이, 다음 조각에는 마침표만 남았습니다. 원문이 짧아도 고정 글자 수로 자르면 의미 없는 조각이 생길 수 있습니다.

### 셀 02 · 크기만 8로 줄이기

중첩은 0으로 두고 같은 원문을 나눠 보세요.

In [2]:
parts = split_text(text, 8, 0)
print(parts)

['도서 대출 기간', '은 14일입니다', '.']


질문에 있는 ‘대출’과 답에 필요한 ‘14일’이 다른 조각으로 갈라졌습니다. 원문 ID 적중률만 같다고 답을 쓸 문맥까지 같은 것은 아닙니다.

## 두 조각이 공유하는 글자 확인하기

겹치는 글자가 있으면 경계에 걸친 문구를 함께 읽을 수 있습니다. 대신 같은 내용이 여러 검색 결과에 중복될 수 있습니다. 크기나 중첩을 바꾸면 그 조각들로 인덱스를 새로 만들어야 합니다.



**볼 곳**　앞 조각의 끝과 뒤 조각의 시작에서 금색 글자가 반복되는 자리를 보세요.

**애니메이션의 의미**　이웃한 두 조각에서 같은 네 글자를 읽어 보세요.


### 과제 03 · 중첩 네 글자 찾기

size=8, overlap=4로 나누어 첫 조각의 마지막 네 글자와 다음 조각의 첫 네 글자가 같은지 확인하세요.

In [3]:
# 같은 원문의 앞 두 조각을 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>overlapped = split_text(text, 8, 4)
print(repr(overlapped[0][-4:]), repr(overlapped[1][:4]))</code></pre>
<p>실행 결과</p><pre><code>&#x27;출 기간&#x27; &#x27;출 기간&#x27;</code></pre>
<p>같은 네 글자가 양쪽에 보입니다. repr을 쓴 까닭은 공백도 한 글자에 들어가기 때문입니다. 다음 실험에서 k만 바꾼다면 이번에 만든 조각과 벡터를 그대로 사용하세요.</p>
</details>

## 같은 순위에서 k만 바꾸기

이번에는 청크를 다시 만들지 않습니다. 37장 실험의 고정 인덱스(size=8, overlap=4)에서 대출 질문으로 찾은 상위 두 청크를 읽습니다. k는 정렬을 바꾸지 않고 앞에서 몇 개를 가져올지 정합니다.



**볼 곳**　‘같은 순위’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘상위 1개’, 아래쪽은 ‘상위 2개’입니다.

**애니메이션의 의미**　k를 늘리면 같은 순위에서 더 많은 후보를 가져옵니다.


### 셀 04 · 한 건에서 두 건으로 늘리기

아래 목록은 제공 실험에서 저장한 dev-01의 순위입니다. 어느 청크가 추가되는지 읽으세요.

추가로 찾은 조각 ‘출 기간은 14’에서 14 다음 글자를 확인하세요. 숫자는 보이지만 단위 ‘일’은 이 조각에 없습니다. 두 건을 찾았다는 사실만으로 답에 필요한 문장이 모두 들어왔다고 판단할 수 없습니다.

In [4]:
ranking = ['library-1:0', 'library-1:1']
print(ranking[:1])
print(ranking[:2])

['library-1:0']
['library-1:0', 'library-1:1']


두 번째 결과가 추가되지만 첫 번째 검색 결과는 그대로입니다. 청크 두 건이 같은 원문에서 왔다면 기대 원문을 두 개 찾은 것으로 세지 않습니다.

## 돌아보기

크기와 k를 동시에 바꿀 경우 어느 쪽 때문에 답이 달라졌는지 나누기 어렵습니다. 다음 실험의 질문·원문·인덱스·k 가운데 고정할 항목을 먼저 적어 보세요.